# 🧠➕🌐 RAG ilovasi — boshidan oxirigacha (to'liq)

Bu notebook **bitta to'liq ilova**: avval hujjatlar bo'yicha javob beradigan RAG tizimini quramiz (`ask()` funksiyasi), so'ng uni **Gradio** yordamida ulashish mumkin bo'lgan veb-ilovaga aylantiramiz.

**Foydalanish tartibi:** yuqoridan pastga barcha katakchalarni ketma-ket ishga tushiring (Shift + Enter). Eng oxirgi katakcha sizga do'stlaringizga yuborsa bo'ladigan **umumiy havola** beradi.

Dasturlashni bilmasangiz ham — har katakcha oldida oddiy izoh bor.

## 1-qadam. Kutubxonalarni o'rnatamiz

- **openai** — OpenAI modellari bilan ishlash.
- **chromadb** — vektorlarni saqlaydigan kichik baza.
- **tiktoken** — matnni tokenlarga bo'lish.
- **gradio** — Python kodidan veb-ilova va ulashiladigan havola yasash.

Bir marta ishga tushirish yetarli.

In [ ]:
!pip install -q "openai>=1.0.0" "chromadb>=0.4.0" tiktoken gradio
print("✅ Kutubxonalar o'rnatildi.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 2-qadam. API kalitni xavfsiz ulaymiz 🔑

Kalitni **hech qachon** ochiq kodga yozmang. **Colab Secrets** ishlatamiz:
1. Chap paneldagi 🔑 belgisini bosing.
2. **+ Add new secret** bosing.
3. **Name:** `OPENAI_API_KEY`, **Value:** kalitingiz.
4. **Notebook access** ni yoqing.

In [ ]:
import os
from openai import OpenAI

OPENAI_API_KEY = None

# 1-usul: Google Colab Secrets (tavsiya etiladi)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

# 2-usul: muhit o'zgaruvchisi (Colab'dan tashqarida ishlasangiz)
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "❌ OPENAI_API_KEY topilmadi!\n\n"
        "Colab Secrets'ga qo'shish uchun:\n"
        "1. Chap paneldagi 🔑 belgisini bosing.\n"
        "2. '+ Add new secret' bosing.\n"
        "3. Name: OPENAI_API_KEY, Value: kalitingiz.\n"
        "4. 'Notebook access' ni yoqing.\n"
        "5. Ushbu katakchani qayta ishga tushiring."
    )

client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ API kalit topildi va OpenAI mijozi tayyor.")

✅ API kalit topildi va OpenAI mijozi tayyor.


## 3-qadam. Hujjatlarni yuklaymiz

Standart holatda 3 ta **demo hujjat** ishlatiladi. O'z fayllaringizni yuklash uchun `USE_UPLOAD = True` qiling.

In [ ]:
documents = []

# --- DEMO HUJJATLAR (aniq faktlar bilan) ---
demo_docs = {
    "quyosh_tizimi.txt": (
        "Quyosh tizimida 8 ta sayyora bor. Yupiter — eng katta sayyora. "
        "Merkuriy — Quyoshga eng yaqin sayyora. Yer — hayot mavjud yagona sayyora. "
        "Mars 'Qizil sayyora' deb ataladi, chunki yuzasi temir oksidiga boy."
    ),
    "bepro_academy.txt": (
        "BePro Academy — dasturlashni o'rgatuvchi o'quv markazi. "
        "RAG-EDU-1 loyihasi — oddiy RAG tizimini yaratish bo'yicha o'quv topshirig'i. "
        "Kurs davomiyligi 6 oy. Darslar o'zbek tilida olib boriladi."
    ),
    "soglik.txt": (
        "Kuniga kamida 8 stakan suv ichish tavsiya etiladi. "
        "Muntazam jismoniy mashqlar yurak-qon tomir kasalliklari xavfini kamaytiradi. "
        "Kattalar uchun kuniga 7-9 soat uyqu me'yor hisoblanadi."
    ),
}

USE_UPLOAD = False  # ← o'z fayllaringizni (.txt / .md) yuklash uchun True qiling

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        if fname.lower().endswith((".txt", ".md")):
            documents.append({"source_file": fname, "text": content.decode("utf-8")})
    print(f"✅ {len(documents)} ta fayl yuklandi.")
else:
    for fname, text in demo_docs.items():
        documents.append({"source_file": fname, "text": text})
    print(f"✅ {len(documents)} ta demo hujjat yuklandi.")

for d in documents:
    print(f"  - {d['source_file']} ({len(d['text'])} belgi)")

✅ 3 ta demo hujjat yuklandi.
  - quyosh_tizimi.txt (204 belgi)
  - bepro_academy.txt (192 belgi)
  - soglik.txt (183 belgi)


## 4-qadam. Matnni bo'laklarga ajratamiz (chunking)

Matnni token bo'yicha kichik **bo'laklarga** ajratamiz. `CHUNK_SIZE` va `CHUNK_OVERLAP` ni bemalol o'zgartirishingiz mumkin.

In [ ]:
import tiktoken

CHUNK_SIZE = 500     # har bir bo'lak necha token
CHUNK_OVERLAP = 75   # qo'shni bo'laklar necha token bilan kesishadi

encoding = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunks.append(encoding.decode(tokens[start:end]))
        if end >= len(tokens):
            break
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "text": piece,
            "source_file": doc["source_file"],
            "chunk_id": f"{doc['source_file']}::chunk_{i}",
            "chunk_index": i,
        })

print(f"✅ Jami {len(all_chunks)} ta chunk yaratildi.")

✅ Jami 3 ta chunk yaratildi.


## 5-qadam. Embedding funksiyasi

Matnni sonlar ro'yxatiga (vektorga) aylantiramiz. Model: `text-embedding-3-small`.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text):
    try:
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
        return resp.data[0].embedding
    except Exception as e:
        raise RuntimeError(
            "❌ Embedding olishda xatolik (noto'g'ri kalit, rate limit yoki internet).\n"
            f"Tafsilot: {e}"
        )

_ = get_embedding(all_chunks[0]["text"])
print("✅ Embedding ishlayapti.")

✅ Embedding ishlayapti.


## 6-qadam. ChromaDB'ga indekslaymiz

> ⚠️ **MUHIM:** bu *in-memory* baza. **Runtime → Restart** qilsangiz indeks o'chadi — katakchalarni qaytadan ishga tushiring.

In [ ]:
import chromadb

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("rag_docs")
except Exception:
    pass

collection = chroma_client.create_collection(name="rag_docs")

print("⏳ Indekslash boshlandi...")
for ch in all_chunks:
    vec = get_embedding(ch["text"])
    collection.add(
        ids=[ch["chunk_id"]],
        embeddings=[vec],
        documents=[ch["text"]],
        metadatas=[{"source_file": ch["source_file"], "chunk_index": ch["chunk_index"]}],
    )

print(f"✅ Indekslash tugadi! {collection.count()} ta chunk saqlandi.")

⏳ Indekslash boshlandi...
✅ Indekslash tugadi! 3 ta chunk saqlandi.


## 7-qadam. `ask()` — asosiy savol-javob funksiyasi

Bu funksiya butun RAG jarayonini bajaradi va **matnli javob** qaytaradi (Gradio shu javobni ko'rsatadi):
1. savolni vektorlashtiradi, 2. eng yaqin `TOP_K` bo'lakni topadi, 3. qat'iy prompt tuzadi, 4. `gpt-4o-mini` bilan javob yozadi, 5. manbalarni qo'shadi.

Agar javob hujjatlarda bo'lmasa — *"Berilgan hujjatlarda bu savolga javob topilmadi"* deydi.

In [ ]:
TOP_K = 4
CHAT_MODEL = "gpt-4o-mini"
TEMPERATURE = 0

def ask(question, top_k=TOP_K):
    """Savolga hujjatlar asosida javob qaytaradi (matn ko'rinishida)."""
    if not question or not question.strip():
        return "Iltimos, savol yozing."

    # 1) Qidiruv (retrieval)
    try:
        q_vec = get_embedding(question)
        results = collection.query(query_embeddings=[q_vec], n_results=top_k)
    except Exception as e:
        return f"❌ Qidiruvda xatolik: {e}"

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    if not docs:
        return "Berilgan hujjatlarda bu savolga javob topilmadi."

    # 2) Kontekst
    context = "\n\n".join(
        f"[Manba: {m['source_file']}]\n{d}" for d, m in zip(docs, metas)
    )

    # 3) Qat'iy prompt
    prompt = (
        "Sen faqat berilgan kontekst asosida javob beradigan yordamchisan.\n"
        "QAT'IY QOIDA: agar javob kontekstda bo'lmasa, aniq shunday yoz: "
        "'Berilgan hujjatlarda bu savolga javob topilmadi'. O'zingdan to'qib chiqarma.\n\n"
        f"KONTEKST:\n{context}\n\n"
        f"SAVOL: {question}\n\n"
        "JAVOB:"
    )

    # 4) Javob generatsiyasi
    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        answer = resp.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ Javob generatsiyasida xatolik: {e}"

    # 5) Manbalar
    sources = sorted(set(m["source_file"] for m in metas))
    return answer + "\n\n📚 Manbalar: " + ", ".join(sources)

print("✅ ask() tayyor.")

✅ ask() tayyor.


## 8-qadam. Tez sinov

Gradio'dan oldin `ask()` ishlayotganini tekshiramiz.

In [ ]:
print(ask("Quyosh tizimidagi eng katta sayyora qaysi?"))

Yupiter — eng katta sayyora.

📚 Manbalar: bepro_academy.txt, quyosh_tizimi.txt, soglik.txt


## 9-qadam. 🌐 Gradio veb-ilova + ulashiladigan havola

Bu oxirgi katakcha. Ishga tushirsangiz:
- savol yoziladigan **katakcha**, tagida **javob**,
- sarlavha va qisqa tavsif,
- 3 ta bosiladigan **namuna savol**,
- va **umumiy havola** (`.gradio.live`) — do'stlaringizga yuboring!

In [ ]:
import gradio as gr
import traceback

def javob_ber(savol):
    if not savol or not savol.strip():
        return "Iltimos, avval savol yozing 🙂"
    try:
        return ask(savol)   # ← yuqorida yaratilgan ask() funksiyamiz
    except Exception as e:
        return "❌ Xatolik:\n" + "".join(traceback.format_exception_only(type(e), e))

ilova = gr.Interface(
    fn=javob_ber,
    inputs=gr.Textbox(label="Savolingiz", placeholder="Bu yerga savolingizni yozing..."),
    outputs=gr.Textbox(label="Javob"),
    title="📚 Hujjatlar bo'yicha savol-javob",
    description="Savol yozing — tizim yuklangan hujjatlar asosida javob beradi. Yoki pastdagi tayyor savollardan birini bosing.",
    examples=[
        ["Quyosh tizimidagi eng katta sayyora qaysi?"],
        ["BePro Academy kursi qancha davom etadi?"],
        ["Kuniga qancha suv ichish tavsiya etiladi?"],
    ],
)

ilova.launch(share=True)   # share=True → umumiy havola YOQILGAN

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://96d4407098135741e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## ✅ Qanday ishlatish va ulashish

1. **Runtime → Run all** (yoki har katakchani yuqoridan pastga Shift+Enter bilan) ishga tushiring.
2. Oxirgi katakcha ikkita havola beradi: mahalliy (`localhost`) va **umumiy** (`https://xxxxx.gradio.live`). **Do'stlaringizga aynan `.gradio.live` havolasini yuboring.**
3. Havolani bosib ilovani yangi oynada ochasiz.

**Eslatmalar (beginner uchun):**
- Havola faqat notebook **ochiq va ishlab turganda** yashaydi. Colab yopilsa yoki uxlab qolsa — havola o'chadi. Qayta ishga tushirsangiz, yangi havola paydo bo'ladi (bepul havola ~72 soat amal qiladi).
- O'z hujjatlaringizni qo'yish uchun: 3-qadamda `USE_UPLOAD = True` qiling va 3→4→5→6-qadamlarni qayta ishga tushiring.
- Namuna savollarni o'z hujjatlaringizga moslab, 9-qadamdagi `examples` ichida o'zgartiring.
- Sozlash: `CHUNK_SIZE`, `CHUNK_OVERLAP` (4-qadam), `TOP_K`, `TEMPERATURE` (7-qadam).